# One-Hot Encoding
**One-Hot (a.k.a. 1-of-K) encoding** is a **standard representational device** that emerged broadly from
**statistics (dummy/indicator variables), pattern recognition, and digital design**.
In machine learning and NLP, the scheme is discussed and used pervasively in canonical sources:

- **Christopher M. Bishop (2006)** — calls it **1-of-K coding** in *Pattern Recognition and Machine Learning*.
- **Trevor Hastie, Robert Tibshirani, Jerome Friedman (2009)** — describe **indicator (dummy) variables** in *The Elements of Statistical Learning*.
- **Ian Goodfellow, Yoshua Bengio, Aaron Courville (2016)** — use **one-hot vectors** for words/characters in *Deep Learning*.
- **C. D. Manning, P. Raghavan, H. Schütze (2008)** — discuss **vector-space text representations** in *Introduction to Information Retrieval* (BoW/binary indicators are the document-level extension of token one-hot).

> In short: no single inventor; one-hot is a foundational encoding convention across statistics and ML.

## Intuitive, Logical Concept (Plain Language)

**One-Hot Encoding** represents a categorical symbol (e.g., a **word token**) by a vector of length **|V|**
(the vocabulary size) with a **single 1** at the index of that token and **0s everywhere else**.

- For example, with vocabulary `[cat, dog, phone, refund]`:
  - `one_hot("dog") = [0, 1, 0, 0]`  
  - `one_hot("refund") = [0, 0, 0, 1]`

### Why it made sense historically
- It gives a **simple, exact identity** for each symbol—no confusion between different words.
- It’s **algebra-friendly**: one-hot vectors are **basis vectors** in $\mathbb{R}^{|V|}$.
- Computation is straightforward and scalable with sparse data structures.

### Limitation (and why embeddings came later)
- **No notion of similarity**: the distance between any two different one-hot vectors is the same.
- **High dimensionality** grows with vocabulary size.
- This motivated **distributed representations (embeddings)** that map similar words to closer vectors in a **much lower-dimensional** space.


### Common Applications

- **Input representation** for classical ML models when vocabularies are small (e.g., intents, product categories).  
- **Inputs to neural networks** (historically): one-hot tokens fed to an **Embedding layer**, which learns dense vectors.  
- **Character-level models**: one-hot over characters (small alphabet) for OCR, language ID, or normalization.  
- **Features for simple rule/linear models**: presence/absence indicators for specific keywords or codes.  
- **Baselines and diagnostics**: a transparent starting point before moving to richer representations.


### Real-World Example: 
Customer Support Intent Classification using One-Hot Encoding

Goal: Classify short customer messages into intents: {"refund", "tech_support", "shipping", "product_info"}.

Method: We'll implement a minimal OneHotVectorizer to convert each document into a *multi-hot* vector
         (document-level OR over token one-hots). This is the standard way one-hot is used for text
         at the document level: each token is one-hot; combined over a document yields a multi-hot vector.

**Notes:**
 - This is a small, self-contained dataset for demonstration. In production, use more data and proper evaluation.
 - We keep tokenization simple and include clear comments (#) as requested.

 Citations for concepts: Bishop (2006; one-of-K coding), Hastie et al. (2009; indicator variables),

 Manning et al. (2008; vector-space text features), Goodfellow et al. (2016; one-hot & embeddings).

### Setup tiny dataset

In [66]:
from __future__ import annotations

import re
from collections import Counter
from typing import List, Iterable, Tuple, Dict

import numpy as np
import pandas as pd

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score
from sklearn.model_selection import train_test_split

# -----------------------------
# 1) Tiny, labeled dataset
# -----------------------------
data: List[Tuple[str, str]] = [
    ("I want a refund for my broken phone", "refund"),
    ("How do I return this item and get a refund?", "refund"),
    ("The product stopped working after a week, refund please", "refund"),
    ("I was charged twice, can I get a refund?", "refund"),
    ("My device won't turn on, need help", "tech support"),
    ("App keeps crashing after the update", "tech support"),
    ("Screen is flickering and battery drains fast", "tech support"),
    ("Bluetooth won't pair with my headphones", "tech support"),
    ("Where is my order? It says delivered but I don't have it", "shipping"),
    ("Tracking hasn't updated in 5 days", "shipping"),
    ("Package arrived damaged, what are my options?", "shipping"),
    ("How long does express shipping take?", "shipping"),
    ("Does this model support wireless charging?", "product info"),
    ("Is there a warranty on this phone?", "product info"),
    ("What is the difference between the Pro and Lite versions?", "product info"),
    ("Do you have this in blue color and 128GB?", "product info"),
    ("Delivery is late and I need it tomorrow", "shipping"),
    ("Camera stopped working, how can I fix it?", "tech support"),
    ("I need to cancel and get my money back", "refund"),
    ("Is there a student discount for this product?", "product info"),
]

df = pd.DataFrame(data, columns=["text", "intent"])
df.head(10)

,text,intent
0,I want a refund for my broken phone,refund
1,How do I return this item and get a refund?,refund
2,"The product stopped working after a week, refu...",refund
3,"I was charged twice, can I get a refund?",refund
4,"My device won't turn on, need help",tech support
5,App keeps crashing after the update,tech support
6,Screen is flickering and battery drains fast,tech support
7,Bluetooth won't pair with my headphones,tech support
8,Where is my order? It says delivered but I don...,shipping
9,Tracking hasn't updated in 5 days,shipping


In [67]:
# -----------------------------
# 2) Minimal tokenizer
# -----------------------------
def tokenize(s: str) -> List[str]:
    # Lowercase and keep only alphabetic tokens (very simple)
    return re.findall(r"[a-zA-Z']+", s.lower())  

### Implementing OneHotVectorizer class

The `fit` function is responsible for **building the vocabulary** of the one-hot vectorizer. It processes the input texts, tokenizes them, removes stopwords, and counts word frequencies. Only words that appear at least `min_freq` times (and within `max_vocab` if specified) are kept. The result is a mapping (`vocab_`) from each selected word to a unique index, along with an inverse mapping (`inv_vocab_`). In short, `fit` **learns which words exist in the dataset and assigns each word an index**.

The `transform` function then takes new input texts and converts them into **one-hot encoded vectors** using the vocabulary built in `fit`. For each text, it creates a binary vector of length equal to the vocabulary size, marking `1` at positions corresponding to the words present in the text, and `0` otherwise. This means `transform` **represents texts numerically in a machine-readable form** while keeping the meaning as "word presence". Together, `fit` and `transform` implement the core mechanism of one-hot encoding for text.

### Code Commentary (Step by Step)

- **Dataset**: Small labeled dataset of short customer messages with four intents (*refund*, *tech_support*, *shipping*, *product_info*).  
- **Tokenizer**: Minimal regex tokenizer lowercases text and keeps alphabetic tokens.  
- **OneHotVectorizer**: Builds document-level **multi-hot** vectors by OR-ing token one-hots.  
- **Classifier**: Linear **LogisticRegression** works well on sparse indicators.  
- **Evaluation**: Accuracy + per-class precision/recall/F1.  
- **Interpretability**: Top-weighted tokens per class show which words drive predictions.  
- **Inference**: Predict intents for new messages.


In [68]:

# 3) One-Hot Vectorizer (document-level multi-hot)
class OneHotVectorizer:
    def __init__(self, min_freq: int = 1, max_vocab: int | None = None, stopwords: Iterable[str] | None = None):
        self.min_freq = min_freq
        self.max_vocab = max_vocab
        self.stopwords = set(stopwords) if stopwords else set()
        self.vocab_: Dict[str, int] = {}
        self.inv_vocab_: List[str] = []
    
    
    def fit(self, texts: Iterable[str]) -> "OneHotVectorizer":
        '''
            This method takes raw texts, counts tokens, filters them, sorts them, applies limits, and builds two dictionaries:
        
            vocab_ (word → index)

            inv_vocab_ (index → word)

            This is the core of one-hot encoding: building a fixed vocabulary.
        '''
        # finding word(token) frequency
        freq = Counter()
        for s in texts:
            tokens = [t for t in tokenize(s) if t not in self.stopwords]
            freq.update(tokens)
        
        # Keeps only tokens that appear at least min_freq times.
        # For example, if min_freq=2, words that appear only once are removed.
        items = [(tok, c) for tok, c in freq.items() if c >= self.min_freq]

        # Sorts vocabulary:
        # First by frequency (-x[1] means descending order).
        # Then alphabetically (x[0]) to break ties.
        # So you get most frequent words first, in a consistent order.
        items.sort(key=lambda x: (-x[1], x[0]))
        
        # If a maximum vocabulary size is set (e.g., 10,000 words), it keeps only the top-N words.
        # Useful for memory and performance.
        if self.max_vocab is not None:
            items = items[: self.max_vocab]
        
        # Builds a dictionary mapping word → index.
        # Example: {'dog': 0, 'cat': 1, 'fish': 2}
        self.vocab_ = {tok: i for i, (tok, _) in enumerate(items)}
        self.inv_vocab_ = [tok for tok, _ in items]
        
        # also compute the multi-hot vectors for given texts and store them
        self.last_vectors_ = self.transform(texts)

        return self

    def vector_df(self, texts: Iterable[str] = None):
        """
        Print the one-hot / multi-hot vectors.
        If texts are provided, it will transform and print them,
        otherwise it prints the last fitted vectors.
        """
        if texts is not None:
            vectors = self.transform(texts)
        else:
            vectors = self.last_vectors_

        vecs = []
        for i, vec in enumerate(vectors):
            vecs.append(vec)
        
        return pd.DataFrame(data=vectors, columns=self.vocab_.keys())
            
    def transform(self, texts: Iterable[str]) -> np.ndarray:
        '''
        The main task of this function is to convert a list of input texts into a One-Hot encoded matrix 
        
        representation using the vocabulary that was built earlier in the fit() function.
        
        '''
        V = len(self.vocab_)
        texts_list = list(texts)
        X = np.zeros((len(texts_list), V), dtype=np.float32)
        for i, s in enumerate(texts_list):
            tokens = set(t for t in tokenize(s) if t in self.vocab_ and t not in self.stopwords)
            for t in tokens:
                X[i, self.vocab_[t]] = 1.0
        return X
    
    def fit_transform(self, texts: Iterable[str]) -> np.ndarray:
        self.fit(texts)
        return self.transform(texts)

STOPWORDS = {
    "the","is","a","an","and","or","to","for","in","on","of","it","this","that","i","my","do","you","have","with","after"
}

### Train and Vectorization

In [69]:
# -----------------------------
# 4) Train / Test split
# -----------------------------
X_train_texts, X_test_texts, y_train, y_test = train_test_split(
    df["text"].tolist(), df["intent"].tolist(), test_size=0.3, random_state=42, stratify=df["intent"]
)

# -----------------------------
# 5) Vectorize
# -----------------------------
vec = OneHotVectorizer(min_freq=0, max_vocab=None, stopwords=STOPWORDS)
X_train = vec.fit_transform(X_train_texts)
X_test = vec.transform(X_test_texts)


### Multi-Hot vectors

In [72]:
df = vec.vector_df()
df

,need,refund,can,get,phone,product,stopped,there,won't,working,...,turn,twice,update,updated,want,warranty,was,week,where,wireless
0,0.0,1.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
5,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,...,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
6,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
7,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
8,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
9,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0


Each row in the dataframe is a multihot encoding representation of a text, and each column shows in which texts the label of each column appears. Each row in the dataframe is a multihot encoding representation of a text. And each column shows in which texts the label of each column appears. The number of these rows is also affected by the 'train-test' operation, so it is less than the number of rows in the original dataset.

In [73]:

# -----------------------------
# 6) Train a linear classifier
# -----------------------------
clf = LogisticRegression(max_iter=2000)
clf.fit(X_train, y_train)

# -----------------------------
# 7) Evaluate
# -----------------------------
pred = clf.predict(X_test)
print("Accuracy:", round(accuracy_score(y_test, pred), 3))
print("\nClassification report:\n", classification_report(y_test, pred))


Accuracy: 0.333

Classification report:
               precision    recall  f1-score   support

product info       0.00      0.00      0.00         2
      refund       1.00      1.00      1.00         1
    shipping       0.00      0.00      0.00         2
tech support       0.20      1.00      0.33         1

    accuracy                           0.33         6
   macro avg       0.30      0.50      0.33         6
weighted avg       0.20      0.33      0.22         6



C:\Users\AbdhM\AppData\Roaming\Python\Python313\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\AbdhM\AppData\Roaming\Python\Python313\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\AbdhM\AppData\Roaming\Python\Python313\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} i

### Inspect top indicative tokens per class

In [74]:
# -----------------------------
# 8) Inspect top indicative tokens per class
# -----------------------------
classes = clf.classes_
coefs = clf.coef_
topn = 8

for k, cls in enumerate(classes):
    idx = np.argsort(coefs[k])[-topn:][::-1]
    print(f"\nTop tokens for class '{cls}':")
    for j in idx:
        tok = vec.inv_vocab_[j]
        w = coefs[k, j]
        print(f"  {tok:>15s}  {w: .3f}")



Top tokens for class 'product info':
            there   0.674
         warranty   0.369
         discount   0.305
          student   0.305
          support   0.303
            model   0.303
             does   0.303
         charging   0.303

Top tokens for class 'refund':
           refund   0.767
              get   0.510
             back   0.308
           cancel   0.308
            money   0.308
             want   0.303
           broken   0.303
           please   0.262

Top tokens for class 'shipping':
             late   0.373
         tomorrow   0.373
         delivery   0.373
          updated   0.342
           hasn't   0.342
         tracking   0.342
             days   0.342
            order   0.271

Top tokens for class 'tech support':
            won't   0.545
            keeps   0.308
         crashing   0.308
              app   0.308
           update   0.308
              how   0.280
           camera   0.280
              fix   0.280


### Predict `intend` on the new data

In [75]:

# -----------------------------
# 9) Use the model on new messages
# -----------------------------
new_msgs = [
    "My order is late and the tracking page is empty",
    "The app crashed again after the patch",
    "Can I return this and get my money back?",
    "Does this laptop support USB-C charging?",
]

X_new = vec.transform(new_msgs)
new_pred = clf.predict(X_new)
data = []
for msg, p in zip(new_msgs, new_pred):
    data.append([msg,p])
    #print(f'"{msg}"  ->  intent: {p}')

df = pd.DataFrame(data=data, columns=['message', 'intend'])

df

,message,intend
0,My order is late and the tracking page is empty,shipping
1,The app crashed again after the patch,tech support
2,Can I return this and get my money back?,refund
3,Does this laptop support USB-C charging?,product info


## Related Original/Canonical Articles & Books

- **Harris, Z. S. (1954). _Distributional Structure_.** *Word*, 10(2–3), 146–162.  
  *Contextual origin of distributional thinking—representing words by observable occurrences.*

- **Salton, G., Wong, A., & Yang, C.-S. (1975). _A Vector Space Model for Automatic Indexing_.** *Communications of the ACM*, 18(11), 613–620.  
  *Classical vector-space formulation for documents; binary/weighted indicators as features.*

- **Bishop, C. M. (2006). _Pattern Recognition and Machine Learning_.** Springer.  
  *Introduces/uses **1-of-K** coding formally in ML.*

- **Hastie, T., Tibshirani, R., & Friedman, J. (2009). _The Elements of Statistical Learning_.** Springer.  
  *Systematic treatment of categorical predictors via indicator (dummy) variables.*

- **Goodfellow, I., Bengio, Y., & Courville, A. (2016). _Deep Learning_.** MIT Press.  
  *Ubiquitous use of one-hot for tokens prior to embeddings; clarifies limitations and transitions to distributed representations.*

- **Manning, C. D., Raghavan, P., & Schütze, H. (2008). _Introduction to Information Retrieval_.** Cambridge University Press.  
  *Vector-space text representations (binary/count/TF–IDF); practical grounding for text features.*

# If this topic was helpful to you, please give me a star ⭐.